<a href="https://colab.research.google.com/github/Kakanan007/3Dkkc/blob/main/%E0%B8%87%E0%B8%B2%E0%B8%99%E0%B8%99%E0%B8%99%E0%B8%99%E0%B8%99%E0%B8%99%E0%B8%99%E0%B8%99%E0%B8%99%E0%B8%99%E0%B8%99%E0%B8%99_Data_mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install streamlit joblib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 78.6 MB/s eta 0:00:00


In [ ]:
!pip install pandas scikit-learn joblib streamlit -q

##ข้อมูล


In [ ]:
import pandas as pd
import numpy as np

# ตั้งค่า Seed เพื่อให้สุ่มได้ผลลัพธ์เดิม
np.random.seed(42)
n_samples = 10000

# 1. สร้างข้อมูลสุ่ม (Synthetic Data)
transaction_ids = [f"TXN{i:06d}" for i in range(1, n_samples + 1)]
amounts = np.random.exponential(scale=2000, size=n_samples) + 50  # สุ่มยอดเงินใช้จ่าย
tx_freq_1h = np.random.poisson(lam=1.5, size=n_samples)          # ความถี่การรูดใน 1 ชม.
distance_km = np.random.exponential(scale=15, size=n_samples)     # ระยะห่างจากพิกัดเดิม

# ช่องทางการทำรายการ และประเภทร้านค้า
channels = np.random.choice(['Online', 'POS_Swipe', 'Contactless'], size=n_samples, p=[0.5, 0.4, 0.1])
merchant_types = np.random.choice(['Retail', 'Travel', 'Dining', 'Digital_Goods', 'Luxury'], size=n_samples)

# ข้อมูลทางเทคนิค
device_ids = [f"DEV_{np.random.randint(1000, 9999)}" for _ in range(n_samples)]
ip_addresses = [f"192.168.{np.random.randint(1, 255)}.{np.random.randint(1, 255)}" for _ in range(n_samples)]

# สถานะ Fraud (ให้มี Fraud ประมาณ 1% ตามเอกสาร)
is_fraud = np.random.choice([0, 1], size=n_samples, p=[0.99, 0.01])

# ปรับให้เคสที่เป็น Fraud มียอดเงิน หรือความถี่สูงผิดปกติ
for i in range(n_samples):
    if is_fraud[i] == 1:
        amounts[i] = amounts[i] * np.random.uniform(3, 8)
        tx_freq_1h[i] = tx_freq_1h[i] + np.random.randint(3, 10)

# 2. รวมข้อมูลเป็น DataFrame
df = pd.DataFrame({
    'transaction_id': transaction_ids,
    'amount': np.round(amounts, 2),
    'tx_freq_1h': tx_freq_1h,
    'distance_km': np.round(distance_km, 2),
    'channel': channels,
    'merchant_type': merchant_types,
    'device_id': device_ids,
    'ip_address': ip_addresses,
    'is_fraud': is_fraud
})

# 3. แทรก Missing Values (ขยะ/ข้อมูลสูญหาย)
# สุ่มลบข้อมูล IP Address, Device ID และ Distance บางส่วน (ประมาณ 5% - 8%)
mask_ip = np.random.rand(n_samples) < 0.05
mask_dev = np.random.rand(n_samples) < 0.05
mask_dist = np.random.rand(n_samples) < 0.08

df.loc[mask_ip, 'ip_address'] = np.nan
df.loc[mask_dev, 'device_id'] = np.nan
df.loc[mask_dist, 'distance_km'] = np.nan

# 4. ส่งออกเป็นไฟล์ CSV และ Excel
df.to_csv("credit_card_fraud_10k.csv", index=False)
df.to_excel("credit_card_fraud_10k.xlsx", index=False)

print("สร้างข้อมูลสำเร็จ! (บันทึกเป็น credit_card_fraud_10k.xlsx และ .csv)")
print(df.info())

สร้างข้อมูลสำเร็จ! (บันทึกเป็น credit_card_fraud_10k.xlsx และ .csv)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  10000 non-null  object 
 1   amount          10000 non-null  float64
 2   tx_freq_1h      10000 non-null  int64  
 3   distance_km     9189 non-null   float64
 4   channel         10000 non-null  object 
 5   merchant_type   10000 non-null  object 
 6   device_id       9497 non-null   object 
 7   ip_address      9514 non-null   object 
 8   is_fraud        10000 non-null  int64  
dtypes: float64(2), int64(2), object(5)
memory usage: 703.3+ KB
None


#clean data

In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. โหลดและ Clean ข้อมูล
df = pd.read_csv("credit_card_fraud_10k.csv")
df['distance_km'] = df.groupby('channel')['distance_km'].transform(lambda x: x.fillna(x.median()))
df['device_id'] = df['device_id'].fillna('Unknown')
df['ip_address'] = df['ip_address'].fillna('Unknown')

# 2. เตรียม Feature
features = ['amount', 'tx_freq_1h', 'distance_km', 'channel', 'merchant_type']

X = pd.get_dummies(df[features], drop_first=True)
y = df['is_fraud']

# 3. เทรนโมเดล
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

joblib.dump(model, 'fraud_model.pkl')
joblib.dump(X.columns.tolist(), 'model_columns.pkl')

['model_columns.pkl']

In [ ]:
!pip install gradio -q

import gradio as gr
import pandas as pd
import joblib

# 1. โหลดโมเดล
model = joblib.load('fraud_model.pkl')
model_columns = joblib.load('model_columns.pkl')

# 2. ฟังก์ชันประเมินความเสี่ยง
def predict_fraud(amount, tx_freq_1h, distance_km, channel, merchant_type):
    input_data = pd.DataFrame([{
        'amount': amount,
        'tx_freq_1h': tx_freq_1h,
        'distance_km': distance_km,
        'channel': channel,
        'merchant_type': merchant_type
    }])

    input_encoded = pd.get_dummies(input_data)
    input_encoded = input_encoded.reindex(columns=model_columns, fill_value=0)

    prob = model.predict_proba(input_encoded)[0][1]

    if prob >= 0.7:
        return f"🚨 ความเสี่ยงสูง (โอกาสทุจริต {prob*100:.1f}%)\n คำสั่งระบบ: อายัดบัตรชั่วคราว และส่งเรื่องให้ทีม Fraud ตรวจสอบ"
    elif prob >= 0.3:
        return f"⚠️ ความเสี่ยงปานกลาง (โอกาสทุจริต {prob*100:.1f}%)\n คำสั่งระบบ: ส่ง SMS ขอรหัส OTP ยืนยันตัวตน"
    else:
        return f"✅ ความเสี่ยงต่ำ (โอกาสทุจริต {prob*100:.1f}%)\n คำสั่งระบบ: อนุมัติทำรายการสำเร็จ"

# 3. สร้าง UI
demo = gr.Interface(
    fn=predict_fraud,
    inputs=[
        gr.Number(label="ยอดเงินที่ใช้ (บาท)", value=1500.0),
        gr.Slider(label="ความถี่การใช้บัตรใน 1 ชม.", minimum=0, maximum=10, value=1),
        gr.Number(label="ระยะห่างจากที่อยู่ประจำ (กม.)", value=5.0),
        gr.Dropdown(label="ช่องทางชำระเงิน", choices=["Online", "POS_Swipe", "Contactless"], value="Online"),
        gr.Dropdown(label="ประเภทร้านค้า", choices=["Dining", "Digital_Goods", "Luxury", "Travel", "Retail"], value="Dining")
    ],
    outputs=gr.Textbox(label="ผลการประเมินความเสี่ยงอัตโนมัติ (Action)"),
    title="💳 ระบบตรวจจับการทุจริตบัตรเครดิต (Prototype)",
    description="ระบบประเมินความเสี่ยงและตัดสินใจอัตโนมัติตามกรอบ CRISP-DM เพื่อลดปัญหาการระงับบัตรผู้บริสุทธิ์",
    flagging_mode="never"
)

# 4. รัน UI
demo.launch(inline=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1d50eef55b85e4214b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
